# Importe

In [15]:
from typing import Dict, Tuple, Set, List, Any
import re
import math
import pandas as pd
from collections import defaultdict, Counter, deque

# Funktionen

In [16]:
##############################
# 1) DFA parsing
##############################

def parse_dfa(text: str) -> Dict[str, Any]:
    """
    Parse DFA text like:
      "A: a-->B, b-->B; B: a-->B, b-->A; in:B; fi:A,B"
    Returns a dict with: states, alphabet, start, finals, transitions
      transitions: dict[(state, symbol)] = target_state
    """
    parts = [p.strip() for p in text.split(";") if p.strip()]
    transitions = {}
    states: Set[str] = set()
    alphabet: Set[str] = set()
    start = None
    finals: Set[str] = set()

    # state line looks like "A: a-->B, b-->B"
    state_line_pattern = re.compile(r"^([A-Za-z0-9_]+)\s*:\s*(.+)$")
    trans_token_pattern = re.compile(r"\s*([^\s,:]+)\s*-->\s*([^\s,;]+)\s*")

    for p in parts:
        if p.startswith("in:"):
            start = p.split("in:", 1)[1].strip()
        elif p.startswith("fi:"):
            finals = set([s.strip() for s in p.split("fi:", 1)[1].split(",") if s.strip()])
        else:
            m = state_line_pattern.match(p)
            if m:
                state = m.group(1).strip()
                states.add(state)
                trans_list = m.group(2).split(",")
                for tok in trans_list:
                    tok = tok.strip()
                    if not tok:
                        continue
                    m2 = trans_token_pattern.match(tok)
                    if m2:
                        sym = m2.group(1).strip()
                        tgt = m2.group(2).strip()
                        transitions[(state, sym)] = tgt
                        states.add(tgt)
                        alphabet.add(sym)

    # Reachability check (optional)
    if start and states:
        q = deque([start])
        visited = set([start])
        while q:
            u = q.popleft()
            # follow all outgoing transitions
            for (s, sym), t in transitions.items():
                if s == u and t not in visited:
                    visited.add(t)
                    q.append(t)
        reachable_states = visited
    else:
        reachable_states = set()

    return {
        "states": states,
        "alphabet": alphabet,
        "start": start,
        "finals": finals,
        "transitions": transitions,
        "reachable_states": reachable_states
    }

In [17]:
##############################
# 2) Feature engineering
##############################

def dfa_features(dfa: Dict[str, Any]) -> Dict[str, Any]:
    states = dfa["states"]
    alphabet = dfa["alphabet"]
    trans = dfa["transitions"]
    finals = dfa["finals"]
    start = dfa["start"]
    reachable = dfa["reachable_states"]

    n_states = len(states)
    alphabet_size = len(alphabet)
    n_transitions = len(trans)
    n_self_loops = sum(1 for (s, a), t in trans.items() if s == t)
    final_ratio = (len(finals) / n_states) if n_states else 0.0

    # out-degree per state (unique symbols with transitions defined)
    out_counts = Counter()
    for (s, a), t in trans.items():
        out_counts[s] += 1
    avg_out_degree = (sum(out_counts.values()) / n_states) if n_states else 0.0

    # completeness
    is_complete = True
    if n_states and alphabet_size:
        for s in states:
            for a in alphabet:
                if (s, a) not in trans:
                    is_complete = False
                    break
            if not is_complete:
                break

    n_reachable = len(reachable) if start else 0
    unreachable = n_states - n_reachable if n_states else 0

    # number of sink states: states whose outgoing transitions (for all symbols present) go to itself
    sink_states = 0
    for s in states:
        # consider only symbols that actually exist in this DFA
        if alphabet_size == 0:
            continue
        all_self = True
        # If a transition is missing, it's not a strict sink in the total sense
        for a in alphabet:
            if (s, a) not in trans or trans[(s, a)] != s:
                all_self = False
                break
        if all_self:
            sink_states += 1

    return {
        "n_states": n_states,
        "alphabet_size": alphabet_size,
        "n_transitions": n_transitions,
        "n_self_loops": n_self_loops,
        "final_ratio": final_ratio,
        "avg_out_degree": avg_out_degree,
        "is_complete": int(is_complete),
        "n_reachable": n_reachable,
        "n_unreachable": unreachable,
        "n_sink_states": sink_states,
    }


def parse_pair_to_features(input_text: str, output_text: str) -> Dict[str, Any]:
    dfa_in = parse_dfa(input_text)
    dfa_out = parse_dfa(output_text)
    feat = dfa_features(dfa_in)
    feat["minimized_n_states"] = len(dfa_out["states"])
    feat["reduction_ratio"] = (
        (feat["minimized_n_states"] / feat["n_states"]) if feat["n_states"] else math.nan
    )
    return feat

In [19]:
##############################
# 3) Rough Set basics
##############################

def discretize_dataframe(df: pd.DataFrame, numeric_cols: List[str], bins: int = 3, strategy: str = "quantile") -> pd.DataFrame:
    """
    Discretize numeric columns into categorical labels.
    strategy: 'quantile' or 'uniform'
    """
    df = df.copy()
    for col in numeric_cols:
        if strategy == "quantile":
            # handle constant columns safely
            if df[col].nunique(dropna=True) <= 1:
                df[col + "_disc"] = 0
            else:
                df[col + "_disc"] = pd.qcut(df[col].rank(method="first"), q=bins, labels=False, duplicates="drop")
        else:  # uniform
            if df[col].nunique(dropna=True) <= 1:
                df[col + "_disc"] = 0
            else:
                df[col + "_disc"] = pd.cut(df[col], bins=bins, labels=False, duplicates="drop")
    return df


def indiscernibility_partition(df: pd.DataFrame, attrs: List[str]) -> List[List[int]]:
    """
    Partition universe U = {0..n-1} into equivalence classes by equality on 'attrs'.
    Returns list of lists of row indices.
    """
    groups = defaultdict(list)
    for i, row in df[attrs].iterrows():
        key = tuple(row.tolist())
        groups[key].append(i)
    return list(groups.values())


def dependency_degree(df: pd.DataFrame, cond_attrs: List[str], decision_attr: str) -> float:
    """
    Pawlak dependency degree gamma(C, D).
    """
    if not cond_attrs:
        return 0.0
    U = set(df.index.tolist())
    part_C = indiscernibility_partition(df, cond_attrs)
    # a block is consistent if all rows share the same decision value
    consistent_size = 0
    for block in part_C:
        dec_vals = set(df.loc[block, decision_attr].tolist())
        if len(dec_vals) == 1:
            consistent_size += len(block)
    return consistent_size / len(U) if len(U) else 0.0


def quick_reduct(df: pd.DataFrame, all_attrs: List[str], decision_attr: str) -> List[str]:
    """
    Simplified QuickReduct algorithm.
    """
    R: List[str] = []
    gamma_star = dependency_degree(df, all_attrs, decision_attr)
    while True:
        best_attr = None
        best_gamma = dependency_degree(df, R, decision_attr)
        for a in all_attrs:
            if a in R:
                continue
            g = dependency_degree(df, R + [a], decision_attr)
            if g > best_gamma + 1e-12:
                best_gamma = g
                best_attr = a
        if best_attr is None or abs(best_gamma - gamma_star) < 1e-12:
            break
        R.append(best_attr)
    return R


def induce_rules(df: pd.DataFrame, reduct_attrs: List[str], decision_attr: str) -> List[Dict[str, Any]]:
    """
    Very simple rule induction:
    - For each equivalence class by reduct_attrs, if it is decision-consistent, emit a rule.
    """
    rules = []
    blocks = indiscernibility_partition(df, reduct_attrs)
    for block in blocks:
        dec_vals = set(df.loc[block, decision_attr].tolist())
        if len(dec_vals) == 1:
            # create a conjunctive rule attr=value for the first row as representative
            i = block[0]
            premise = {a: df.at[i, a] for a in reduct_attrs}
            decision = list(dec_vals)[0]
            support = len(block)
            rules.append({"premise": premise, "decision": decision, "support": support})
    return rules

In [30]:
def quick_reduct(df: pd.DataFrame, all_attrs: list, decision_attr: str) -> list:
    R = []
    gamma_R = 0.0
    gamma_star = dependency_degree(df, all_attrs, decision_attr)

    while gamma_R < gamma_star - 1e-12:  # solange wir die volle Abhängigkeit noch nicht erreicht haben
        best_attr = None
        best_gamma = gamma_R

        for a in all_attrs:
            if a in R:
                continue
            g = dependency_degree(df, R + [a], decision_attr)
            if g > best_gamma + 1e-12:
                best_gamma = g
                best_attr = a

        if best_attr is None:
            # keine Verbesserung mehr möglich
            break

        R.append(best_attr)
        gamma_R = best_gamma

    return R


# Ausführung

In [23]:
##############################
# 4) Demo on toy examples
##############################

toy_pairs = [
    # Example from the prompt
    (
        "A: a-->B, b-->B; B: a-->B, b-->A; in:B; fi:A,B",
        "A_B: a-->A_B, b-->A_B; in:A_B; fi:A_B",
    ),
    # Another small DFA where no reduction occurs (already minimal)
    (
        "A: a-->B, b-->A; B: a-->A, b-->B; in:A; fi:B",
        "A: a-->B, b-->A; B: a-->A, b-->B; in:A; fi:B",
    ),
    # A DFA with a dead sink state S all transitions to S
    (
        "A: a-->S, b-->S; S: a-->S, b-->S; in:A; fi:",
        "A: a-->S, b-->S; S: a-->S, b-->S; in:A; fi:",
    ),
]

rows = []
for inp, out in toy_pairs:
    rows.append(parse_pair_to_features(inp, out))

df = pd.DataFrame(rows)

print(df)

   n_states  alphabet_size  n_transitions  n_self_loops  final_ratio  \
0         2              2              4             1          1.0   
1         2              2              4             2          0.5   
2         2              2              4             2          0.0   

   avg_out_degree  is_complete  n_reachable  n_unreachable  n_sink_states  \
0             2.0            1            2              0              0   
1             2.0            1            2              0              0   
2             2.0            1            2              0              1   

   minimized_n_states  reduction_ratio  
0                   1              0.5  
1                   2              1.0  
2                   2              1.0  


In [29]:
# Choose decision as discretized reduction ratio (e.g., 3 classes)
numeric_cols = ["n_states", "alphabet_size", "n_transitions", "n_self_loops",
                "final_ratio", "avg_out_degree", "n_reachable", "n_unreachable",
                "n_sink_states", "reduction_ratio", "minimized_n_states"]

df_disc = discretize_dataframe(df, numeric_cols=numeric_cols, bins=3, strategy="quantile")
# Use discretized features (excluding discretized decision itself) as conditions
cond_attrs = [c for c in df_disc.columns if c.endswith("_disc") and c != "reduction_ratio_disc"]
decision_attr = "reduction_ratio_disc"

# Compute a quick reduct and induce simple rules
reduct = quick_reduct(df_disc, cond_attrs, decision_attr)
rules = induce_rules(df_disc, reduct, decision_attr)

# Display the discretized decision table
print(df_disc.head())
print(reduct)
print(rules)

   n_states  alphabet_size  n_transitions  n_self_loops  final_ratio  \
0         2              2              4             1          1.0   
1         2              2              4             2          0.5   
2         2              2              4             2          0.0   

   avg_out_degree  is_complete  n_reachable  n_unreachable  n_sink_states  \
0             2.0            1            2              0              0   
1             2.0            1            2              0              0   
2             2.0            1            2              0              1   

   ...  alphabet_size_disc  n_transitions_disc  n_self_loops_disc  \
0  ...                   0                   0                  0   
1  ...                   0                   0                  1   
2  ...                   0                   0                  2   

   final_ratio_disc  avg_out_degree_disc  n_reachable_disc  \
0                 2                    0                 0   
1

In [31]:
# Compute a quick reduct and induce simple rules
reduct = quick_reduct(df_disc, cond_attrs, decision_attr)
rules = induce_rules(df_disc, reduct, decision_attr)

# Display the discretized decision table
print(df_disc.head())
print(reduct)
print(rules)

   n_states  alphabet_size  n_transitions  n_self_loops  final_ratio  \
0         2              2              4             1          1.0   
1         2              2              4             2          0.5   
2         2              2              4             2          0.0   

   avg_out_degree  is_complete  n_reachable  n_unreachable  n_sink_states  \
0             2.0            1            2              0              0   
1             2.0            1            2              0              0   
2             2.0            1            2              0              1   

   ...  alphabet_size_disc  n_transitions_disc  n_self_loops_disc  \
0  ...                   0                   0                  0   
1  ...                   0                   0                  1   
2  ...                   0                   0                  2   

   final_ratio_disc  avg_out_degree_disc  n_reachable_disc  \
0                 2                    0                 0   
1